# CNDE NDT Document QA — Grounded RAG with Automated Verification
Run the sections **in order** on a fresh Colab runtime with a **T4 GPU** (`Runtime > Change runtime type`).
Reusable logic lives in `src/`; this notebook only wires it together.

## 1. Environment setup

In [ ]:
import sys, platform, shutil, subprocess
print("Python", sys.version.split()[0], "|", platform.platform())
if shutil.which("nvidia-smi"):
    print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv"],
                         capture_output=True, text=True).stdout)
else:
    print("NO GPU on this runtime -> switch to a T4 GPU runtime before section 5.")

## 2. Dependency installation
Uses the code synced by the Colab folder-sync extension (`/content/cnde`) if present; otherwise clones the public repo. torch is preinstalled on Colab and is not reinstalled.

In [ ]:
import os
REPO_URL = "https://github.com/AbhijeetKumar1505/CNDE.git"
BRANCH = "main"                # or a feature branch, e.g. "feature/llm-verification"
SYNCED_DIR = "/content/cnde"    # local folder synced by the Colab sync extension
CLONE_DIR = "/content/CNDE"

if os.path.isdir(os.path.join(SYNCED_DIR, "src")):
    REPO_DIR = SYNCED_DIR
    print("Using synced working copy:", REPO_DIR)
else:
    REPO_DIR = CLONE_DIR
    if not os.path.isdir(REPO_DIR):
        !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
    else:
        !git -C {REPO_DIR} fetch origin && git -C {REPO_DIR} checkout {BRANCH} && git -C {REPO_DIR} pull origin {BRANCH}
    !git -C {REPO_DIR} log --oneline -3
%cd {REPO_DIR}
%pip install -q -r requirements.txt

## 3. Google Drive mounting
Persistent storage: `MyDrive/CNDE_Hackathon/`. If mounting is unavailable (e.g. VS Code Colab kernel), `Config` falls back to `<repo>/workspace` on the runtime, which does not persist across runtimes.

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as e:  # report, don't hide: storage falls back to the runtime-local workspace
    print(f"Drive NOT mounted ({type(e).__name__}: {e}). Using runtime-local workspace instead.")

## 4. Configuration

In [ ]:
import sys, importlib
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

def reload_src():
    """Reload project modules so code pulled in section 2 takes effect without restarting the runtime."""
    names = sorted(n for n in sys.modules if n == "src" or n.startswith("src."))
    for name in names:
        importlib.reload(sys.modules[name])
    return names

reloaded = reload_src()   # no-op on a fresh runtime; after a re-run of section 2 this picks up the new code
print("reloaded modules:", reloaded or "none (fresh runtime)")

from src.config import Config
cfg = Config()
cfg.make_dirs()
cfg

## 5. Model loading
Qwen3-4B-Instruct-2507 in 4-bit NF4 (double quant, FP16 compute). The smoke test checks for FP16 overflow (non-finite logits) and records VRAM and decode speed.

In [ ]:
from src.generation import load_llm, smoke_test, gpu_memory
if "model" not in globals():          # load once per runtime: a second load would double VRAM use
    model, tokenizer = load_llm(cfg.llm_id)
else:
    print("model already loaded:", gpu_memory())
smoke = smoke_test(model, tokenizer)
smoke

## 6. Document upload

In [ ]:
import requests, pymupdf
from pathlib import Path

# Public-domain corpus. FAA publishes AC 43.13-1B only as one 646-page PDF; Chapter 5 (NDI) is physical
# pages 222-271 and is extracted below, so citations refer to pages of the extracted file.
NASA_URL = "https://standards.nasa.gov/sites/default/files/standards/NASA/C/0/2023-08-03-NASA-STD-5009C-Approved.pdf"
FAA_URL = "https://www.faa.gov/documentLibrary/media/Advisory_Circular/AC_43.13-1B_w-chg1.pdf"

def download(url, dest):
    r = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=180)
    r.raise_for_status()
    Path(dest).write_bytes(r.content)

nasa, faa_ch5 = cfg.docs_dir / "NASA-STD-5009C.pdf", cfg.docs_dir / "FAA_AC_43.13-1B_Ch5_NDI.pdf"
if not nasa.exists():
    download(NASA_URL, nasa)
if not faa_ch5.exists():
    full = cfg.root / "documents" / "source" / "AC_43.13-1B_w-chg1.pdf"   # full AC kept outside docs_dir
    full.parent.mkdir(parents=True, exist_ok=True)
    if not full.exists():
        download(FAA_URL, full)
    src, out = pymupdf.open(full), pymupdf.open()
    out.insert_pdf(src, from_page=221, to_page=270)   # 0-based -> physical pages 222..271
    out.save(str(faa_ch5))

# Add your own PDF/DOCX/TXT files to this folder (or upload them in the Gradio app).
doc_paths = sorted(p for p in cfg.docs_dir.iterdir() if p.suffix.lower() in {".pdf", ".docx", ".txt"})
for p in doc_paths:
    print(f"{p.name:40s} {p.stat().st_size / 1e6:6.2f} MB")

## 7. Text extraction

In [ ]:
import time
from src.ingestion import extract_document

extracted = {}
for p in doc_paths:
    t0 = time.perf_counter()
    extracted[p.name] = pages = extract_document(p)
    empty = [pg["page"] for pg in pages if not pg["blocks"]]
    print(f"{p.name}: {len(pages)} pages, {sum(len(pg['blocks']) for pg in pages)} blocks, "
          f"pages without usable text {empty} ({time.perf_counter() - t0:.2f}s)")

sample = extracted[doc_paths[0].name][0]
print(f"\nSample: page {sample['page']} of {sample['source']}")
for b in sample["blocks"][:6]:
    print(" -", (f"[heading: {b['heading']}] " if b["heading"] else "") + b["text"][:150])

## 8. Chunking

In [ ]:
import statistics
from src.ingestion import chunk_pages
from src.retrieval import load_embedder, token_counter

embedder = load_embedder(cfg.embed_id, cfg.embed_device)   # its tokenizer defines the chunk budget
count_tokens = token_counter(embedder)
t0 = time.perf_counter()
chunks = [c for pages in extracted.values()
          for c in chunk_pages(pages, count_tokens, cfg.chunk_tokens, cfg.chunk_overlap)]
assert len({c["chunk_id"] for c in chunks}) == len(chunks), "duplicate chunk ids"
n = [c["n_tokens"] for c in chunks]
print(f"{len(chunks)} chunks in {time.perf_counter() - t0:.1f}s | BGE tokens min/median/max "
      f"{min(n)}/{statistics.median(n)}/{max(n)} | embedder on {embedder.device}")
chunks[len(chunks) // 2]

## 9. Embedding generation

In [ ]:
from src.retrieval import build_index

t0 = time.perf_counter()
index = build_index(chunks, embedder)   # normalized BGE vectors -> FAISS inner product == cosine
print(f"Embedded {index.ntotal} chunks ({index.d}-d) in {time.perf_counter() - t0:.1f}s on {embedder.device}")

## 10. FAISS indexing

In [ ]:
from src.retrieval import load_index, retrieve, save_index

save_index(index, chunks, cfg.index_dir,
           meta={"embed_id": cfg.embed_id, "chunk_tokens": cfg.chunk_tokens, "chunk_overlap": cfg.chunk_overlap})
index, chunks = load_index(cfg.index_dir, expected_embed_id=cfg.embed_id)   # round-trip check
print(f"Saved + reloaded: {index.ntotal} vectors == {len(chunks)} chunks -> {cfg.index_dir}")

for q in ["What should be done with magnetic particles after a magnetic particle inspection?",
          "What crack sizes can standard penetrant NDE reliably detect?",
          "What is the boiling point of mercury?"]:
    t0 = time.perf_counter()
    hits = retrieve(q, index, chunks, embedder, cfg.top_k)
    print(f"\nQ: {q}  ({(time.perf_counter() - t0) * 1000:.0f} ms)")
    for h in hits[:3]:
        print(f"  {h['score']:.3f} | {h['source']} p.{h['page']} | {h['section'][:60]}")

## 11. Question answering

In [ ]:
from src.generation import gpu_memory
from src.pipeline import answer_question

DEMO_QUESTIONS = [
    "What should be done with magnetic particles after a magnetic particle inspection?",
    "What crack sizes can standard penetrant NDE reliably detect?",
    "What is the boiling point of mercury?",   # out of scope: expect abstention
]
results = []
for q in DEMO_QUESTIONS:
    r = answer_question(q, index, chunks, embedder, model, tokenizer, cfg, variant="v2")
    results.append(r)
    print(f"Q: {q}\n  retrieval {r['retrieval_ms']} ms | generation {r['latency_s']} s | prompt {r['prompt_tokens']} tok"
          f" | new {r['new_tokens']} tok | gated={r['gated']} | hit_max={r['hit_max_tokens']}")
    print("  RAW:", r["answer"][:1000], "\n")
print(gpu_memory())

## 12. Evidence verification

In [ ]:
from IPython.display import Markdown, display
from src.pipeline import to_markdown

for q, r in zip(DEMO_QUESTIONS, results):
    display(Markdown(f"---\n## Q: {q}\n" + to_markdown(r)))
    for c in r["verification"]["claims"]:
        print(f"  claim ok={c['supported']} sem={c['semantic']} lex={c['lexical']} cited={c['cited']} | {c['text'][:90]}")

## 13. Evaluation

In [ ]:
import json, time, faiss, pandas as pd
from IPython.display import display
from src.evaluation import check_benchmark, load_benchmark, retrieval_metrics, run_benchmark, summarize
from src.ingestion import ingest_files
from src.pipeline import answer_question
from src.retrieval import add_to_index

# Evaluation index = corpus + two clearly labelled SYNTHETIC documents (a conflicting value and a
# prompt-injection attempt). Kept separate from the demo index.
syn_paths = sorted((Path(REPO_DIR) / "benchmarks" / "synthetic").glob("*.txt"))
syn_chunks, _ = ingest_files(syn_paths, count_tokens, cfg.chunk_tokens, cfg.chunk_overlap)
eval_index, eval_chunks = faiss.clone_index(index), list(chunks)
add_to_index(eval_index, eval_chunks, syn_chunks, embedder)

questions = load_benchmark(cfg.benchmark_file)
problems = check_benchmark(questions, eval_chunks)
print(f"{len(questions)} hand-written questions | benchmark self-check:", problems or "OK")

ret_rows, ret_summary = retrieval_metrics(questions, eval_index, eval_chunks, embedder, ks=(1, 3, 5))
print("Retrieval:", ret_summary)

VARIANTS = ["v1", "v2", "v3"]        # same questions, same retrieved evidence, only the prompt changes
t0 = time.perf_counter()
rows = run_benchmark(questions, lambda q, v: answer_question(q, eval_index, eval_chunks, embedder, model,
                                                             tokenizer, cfg, v), VARIANTS)
summary = summarize(rows)
print(f"\nBenchmark finished in {(time.perf_counter() - t0) / 60:.1f} min | {gpu_memory()}")
display(pd.DataFrame(summary).T)
display(pd.DataFrame(rows).pivot(index="id", columns="variant", values="correct"))

out = cfg.results_dir / f"benchmark_{time.strftime('%Y%m%d_%H%M')}.json"
out.write_text(json.dumps({"retrieval": ret_summary, "retrieval_rows": ret_rows, "summary": summary, "rows": rows,
                           "config": {k: str(v) for k, v in vars(cfg).items()}}, indent=1, ensure_ascii=False))
print("saved:", out)
print(json.dumps({"retrieval": ret_summary, "summary": summary}, indent=1))   # kept in the notebook output

## 14. Demo execution

In [ ]:
# TODO: Milestone 5 — Gradio app